# NECOFS WAVE Forecast Visualization

Opens the WAVE Icechunk store directly and visualizes wave fields with xugrid + hvplot.

In [ ]:
import os

import icechunk
import xarray as xr
import xugrid as xu
import hvplot.xugrid
from dotenv import load_dotenv

load_dotenv(os.path.expanduser('~/dotenv/gom3_forecast.env'))

## Open Icechunk store

In [ ]:
BUCKET    = 'neracoos-necofs-forecast'
REGION    = 'us-east-1'
IC_PREFIX = 'WAVE/icechunk/wave_forecast.icechunk'

config = icechunk.RepositoryConfig.default()
config.set_virtual_chunk_container(
    icechunk.VirtualChunkContainer(
        url_prefix=f's3://{BUCKET}/',
        store=icechunk.s3_store(region=REGION),
    ),
)
storage     = icechunk.s3_storage(bucket=BUCKET, prefix=IC_PREFIX, region=REGION)
credentials = icechunk.containers_credentials({f's3://{BUCKET}/': icechunk.s3_credentials()})
repo        = icechunk.Repository.open(storage, config, authorize_virtual_chunk_access=credentials)
session     = repo.readonly_session('main')
ds = xr.open_zarr(session.store, consolidated=False, chunks={})
ds

In [ ]:
ds['hs']

## Create xugrid dataset

In [ ]:
uds = xu.UgridDataset(ds)
uds

## Visualize significant wave height

In [ ]:
uds['hs'].hvplot.trimesh(
    title='Significant Wave Height (m)',
    geo=True,
    rasterize=True,
    cmap='viridis',
    clabel='Hs (m)',
    tiles='CartoLight',
    frame_width=700,
)